# `d3rlpy` tutorial

In [ ]:
import d3rlpy
from d3rlpy.datasets import get_cartpole
from d3rlpy.algos import DQNConfig, TD3Config
from d3rlpy.metrics import TDErrorEvaluator, EnvironmentEvaluator
import gymnasium as gym
from pmind.replay import convert_rb_to_dataset, mix_datasets_transitions

import numpy as np

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Getting started

In [31]:
# Prepare dataset
# Dataset = replay buffer. We can use integrated dataset or create our own!
dataset, env = get_cartpole()

# Setup algorithm
dqn = DQNConfig().create(device=None)
# NOTE: we can put most of cfg here!

# Initialize NN with right obs and action dims
dqn.build_with_dataset(dataset) 

# Setup metrics

# This metric suggests how Q functions overfit to training sets. 
# If the TD error is large, the Q functions are overfitting.
td_error_evaluator = TDErrorEvaluator(episodes=dataset.episodes)

env_evaluator = EnvironmentEvaluator(env)

rewards = env_evaluator(dqn, dataset=None)
print(f"Reward at initialization: {rewards}")

2026-04-29 20:41.20 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('int32')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(4,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2026-04-29 20:41.20 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.DISCRETE: 2>
2026-04-29 20:41.20 [info     ] Action size has been automatically determined. action_size=2
Reward at initialization: 70.6


/Users/vlad/Documents/University/Master-MIND/projet-mind/.venv/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


In [146]:
# Offline training
dqn.fit(
    dataset,
    n_steps=10000,
    evaluators={
        'td_error': td_error_evaluator,
        'environment': env_evaluator
    }
    )

2026-05-01 14:57.25 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(4,)]), action_signature=Signature(dtype=[dtype('int32')], shape=[(1,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.DISCRETE: 2>, action_size=2)
2026-05-01 14:57.25 [warning  ] Skip building models since they're already built.
2026-05-01 14:57.25 [info     ] Directory is created at d3rlpy_logs/DQN_20260501145725
2026-05-01 14:57.25 [info     ] Parameters                     params={'observation_shape': [4], 'action_size': 2, 'config': {'type': 'dqn', 'params': {'batch_size': 32, 'gamma': 0.99, 'observation_scaler': {'type': 'none', 'params': {}}, 'action_scaler': {'type': 'none', 'params': {}}, 'reward_scaler': {'type': 'none', 'params': {}}, 'compile_graph': False, 'learning_rate': 6.25e-05, 'optim_factory': {'type': 'adam', 'params': {'clip_grad_norm': None, 'lr_scheduler_factory': 

Epoch 1/1:   0%|          | 0/10000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [14]:
# Use trained policy
observation, _ = env.reset()
action = dqn.predict(observation.reshape(1,-1)) # don't forget to add batch dimension
value = dqn.predict_value(observation.reshape(1,-1), action)

In [15]:
# Save and load
if False:
    # save full parameters and configurations in a single file.
    dqn.save('dqn.d3')
    # load full parameters and build algorithm
    dqn2 = d3rlpy.load_learnable("dqn.d3")

    # save full parameters only
    dqn.save_model('dqn.pt')
    # load full parameters with manual setup
    dqn3 = DQNConfig().create(device=None)
    dqn3.build_with_dataset(dataset)
    dqn3.load_model('dqn.pt')

    # save the greedy-policy as TorchScript
    dqn.save_policy('policy.pt')
    # save the greedy-policy as ONNX
    # dqn.save_policy('policy.onnx')

# Data collection

Not really relevant for us, since it doesn't provide uniform exploration and we better convert our replay buffers to their format

# Create our dataset

Note that whereas valid trajectories are pretty straightforward to convert to `d3rlpy` replay buffer, the uniform sampling results in "jumps" and needs a special treatment - we should organize them as episodes of sie 1 I guess.

In [16]:
# vector observation
# 1000 steps of observations with shape of (100,)
observations = np.random.random((1000, 100))

# 1000 steps of actions with shape of (4,)
actions = np.random.random((1000, 4))

# 1000 steps of rewards
rewards = np.random.random(1000)

# 1000 steps of terminal flags
# terminals = np.random.randint(2, size=1000)
terminals = np.zeros(1000)
timeouts = np.random.randint(2, size=1000)

dataset = d3rlpy.dataset.MDPDataset(
    observations=observations,
    actions=actions,
    rewards=rewards,
    terminals=terminals,
    timeouts=timeouts,
)

2026-04-27 17:09.14 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float64')], shape=[(4,)]) observation_signature=Signature(dtype=[dtype('float64')], shape=[(100,)]) reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)])
2026-04-27 17:09.14 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2026-04-27 17:09.14 [info     ] Action size has been automatically determined. action_size=4


# Pre / post processing

We are interested only for action pre/post processing for `Pendulum` to normalize action to `[-1.0, 1.0]` and then denormalize it back. 

Do we need to scale observations (we didn't do it in BBRL)?

We won't preprocess rewards neither.

In [17]:
action_scaler = d3rlpy.preprocessing.MinMaxActionScaler(
    minimum=np.array(-2.0),
    maximum=np.array(2.0),
)
# NOTE: we can deduce min max from cfg.action_scaling

sac = d3rlpy.algos.SACConfig(action_scaler=action_scaler).create()

# Customize NN

You can define a custom NN with `torch.nn.Module` or use their `VectorEncoderFactory` for an MLP.

In [18]:
# Can do encoder factory
encoder_factory = d3rlpy.models.VectorEncoderFactory(
    hidden_units=[300, 400],
    activation="tanh",
)
# NOTE: we can put cfg.algorithm.architecture.[actor/critic]_hidden_size
td3 = d3rlpy.algos.TD3Config(
    actor_encoder_factory=encoder_factory, critic_encoder_factory=encoder_factory
).create()


# Online RL & Fine-tuning

No interest.

# Use trained policies

In [19]:
# # save d3 file
# cql_old.save("model.d3")
# # reconstruct full setup from a d3 file
# cql = d3rlpy.load_learnable("model.d3")


# # Option 2: Load pt file

# # save pt file
# cql_old.save_model("model.pt")
# # setup algorithm manually
# cql = d3rlpy.algos.CQLConfig().create()

# # choose one of three to build PyTorch models

# # if you have MDPDataset object
# cql.build_with_dataset(dataset)
# # or if you have Gym-styled environment object
# cql.build_with_env(env)
# # or manually set observation shape and action size
# cql.create_impl((3,), 1)

# # load pretrained model
# cql.load_model("model.pt")

For inference, don't forget the batch dimension.

In [ ]:
dqn.predict(env.observation_space.sample().reshape(1,-1))
# TODO: but it uses the last policy and not the best one?
# TODO: make a `model` object with the same interface that we used for BBRL

array([0])

# Sandbox

In [ ]:

# Failed attempts to mix replay buffers by hand:

# def get_from_dataset(dataset, var_name):
#     var = []
#     for episode in dataset.episodes:
#         if var_name == "terminals":
#             terminals = np.zeros(len(episode))
#             if episode.terminated:
#                 terminals[-1] = 1
#             var.extend(terminals)
#         elif var_name == "timeouts":
#             timeouts = np.zeros(len(episode))
#             if not episode.terminated:
#                 timeouts[-1] = 1
#             var.extend(timeouts)
#         else:
#             var.extend(getattr(episode, var_name))
#     return np.array(var)


# def mix_datasets_transitions(dataset_a, dataset_b, n_samples_a, n_samples_b):
#     obs_a = get_from_dataset(dataset_a, "observations")
#     act_a = get_from_dataset(dataset_a, "actions")
#     rew_a = get_from_dataset(dataset_a, "rewards")
#     term_a = get_from_dataset(dataset_a, "terminals")
#     time_a = get_from_dataset(dataset_a, "timeouts")

#     obs_b = get_from_dataset(dataset_b, "observations")
#     act_b = get_from_dataset(dataset_b, "actions")
#     rew_b = get_from_dataset(dataset_b, "rewards")
#     term_b = get_from_dataset(dataset_b, "terminals")
#     time_b = get_from_dataset(dataset_b, "timeouts")

#     idx_a = np.random.choice(len(obs_a), n_samples_a, replace=False)
#     idx_b = np.random.choice(len(obs_b), n_samples_b, replace=False)

#     observations = np.concatenate([obs_a[idx_a], obs_b[idx_b]])
#     actions = np.concatenate([act_a[idx_a], act_b[idx_b]])
#     rewards = np.concatenate([rew_a[idx_a], rew_b[idx_b]])
#     terminals = np.concatenate([term_a[idx_a], term_b[idx_b]]).astype(bool)
#     # timeouts = np.concatenate([time_a[idx_a], time_b[idx_b]])
#     # Split into one-transition episodes
#     timeouts = (np.arange(n_samples_a + n_samples_b) % 2).astype(bool)
#     timeouts = timeouts & (~terminals)  # if terminated, then it's not timeout
#     # TODO: need to mix them or they get shuffled anyway? hope so...

#     return d3rlpy.dataset.MDPDataset(
#         observations=observations,
#         actions=actions,
#         rewards=rewards,
#         terminals=terminals,
#         timeouts=timeouts,
#     )


# def get_vars_from_transitions(transitions):
#     observations = transitions.observations
#     actions = transitions.actions
#     rewards = transitions.rewards
#     terminals = transitions.terminals
#     timeouts = np.ones(len(transitions)).astype(bool)

#     return observations, actions, rewards, terminals, timeouts


# def mix_datasets_transitions(dataset_a, dataset_b, n_samples_a, n_samples_b):

#     transitions_a = dataset_a.sample_transition_batch(n_samples_a)
#     transitions_b = dataset_a.sample_transition_batch(n_samples_b)

#     obs_a, act_a, rew_a, term_a, time_a = get_vars_from_transitions(transitions_a)
#     obs_b, act_b, rew_b, term_b, time_b = get_vars_from_transitions(transitions_b)

#     observations = np.concatenate([obs_a, obs_b])
#     actions = np.concatenate([act_a, act_b])
#     rewards = np.concatenate([rew_a, rew_b])
#     terminals = np.concatenate([term_a, term_b]).astype(bool)
#     timeouts = np.concatenate([time_a, time_b])
#     timeouts = timeouts & (~terminals)  # if terminated, then it's not timeout
#     print((timeouts & terminals).any())

#     return d3rlpy.dataset.MDPDataset(
#         observations=observations,
#         actions=actions,
#         rewards=rewards,
#         terminals=terminals,
#         timeouts=timeouts,
#     )
